In [ ]:
# Demo 3: Test transcript retrieval/cleaning/chunking

# Use to get hugging face models
!pip install transformers accelerate huggingface_hub
# Use python library to get youtube transcript
!pip install youtube-transcript-api

In [ ]:
# python library to get youtube transcript
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import pipeline

# Load Qwen model (small free version)
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype="auto",
    device_map="auto",
    max_new_tokens=100
)

In [ ]:
# Data sample for claim extraction. (Ex: chunk of transcript if already cleaned).
# sample_data = "This new Minecraft update causes mining speed to decrease by 10% which will be noticed by competitive builders. Yet their new forest biome offers more materials than the winter biome. I think after 40 hours of playing this update is at least better than the second update "

# Test retrieve youtube video transcript
#video_id = "-1Sy6iz43vg" # sample 1 video ID
video_id = "TT4HTDYhiOU" # sample 2 video ID
yta = YouTubeTranscriptApi()
transcript = yta.fetch(video_id)
print(f"Transcript:\n{transcript}")

Transcript:
FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='So guys, Minecraft 26.0 is finally out,', start=0.08, duration=5.04), FetchedTranscriptSnippet(text='and this update brings more than 50', start=3.2, duration=4.079), FetchedTranscriptSnippet(text='changes. Some are small fixes, some are', start=5.12, duration=4.08), FetchedTranscriptSnippet(text='visual upgrades, and some actually', start=7.279, duration=3.761), FetchedTranscriptSnippet(text="change how the game feels. Let's start", start=9.2, duration=3.28), FetchedTranscriptSnippet(text='with the menu. You probably remember', start=11.04, duration=3.28), FetchedTranscriptSnippet(text='when Mojang moved the settings button', start=12.48, duration=3.6), FetchedTranscriptSnippet(text='down in the main menu. A lot of players', start=14.32, duration=3.6), FetchedTranscriptSnippet(text="didn't like that change, and people even", start=16.08, duration=3.76), FetchedTranscriptSnippet(text='started making jokes about the 

Output Format:

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='my little brother recently got Minecraft', start=0.04, duration=3.639), ...

Format Analysis:

- FetchedTranscript object (holds snippets list)
  - snippets list (holds FetchedTranscriptSnippet objects (each object is a snippet))
    - FetchedTranscriptSnippet object (holds 3 attributes: text, start, duration)

In [ ]:
# Test clean raw transcript

# Get only text from each snippet
texts = [snippet.text for snippet in transcript.snippets]

cleaned = [] # list for cleaned snippets
# Loop to remove captions
for text in texts:
  while "[" in text:
    start = text.index("[")
    end = text.index("]") + 1
    caption = text[start:end]
    text = text.replace(caption, "")
  cleaned.append(text) # text becomes element in cleaned list

# Test print cleaned list
print(f"Transcript Text:\n{cleaned}")

Transcript Text:
['So guys, Minecraft 26.0 is finally out,', 'and this update brings more than 50', 'changes. Some are small fixes, some are', 'visual upgrades, and some actually', "change how the game feels. Let's start", 'with the menu. You probably remember', 'when Mojang moved the settings button', 'down in the main menu. A lot of players', "didn't like that change, and people even", 'started making jokes about the browse', 'add-ons option being more important than', 'settings. Well, Mojang listened. The', 'settings button is now back at the top', 'where it feels natural. And not just', 'that, when you open settings now, the', 'entire layout has been redesigned. It', 'looks cleaner, more modern, and much', 'easier to navigate. The categories are', 'better organized, and overall, it just', "feels more polished. Right now, it's in", 'beta, but soon this redesign should come', 'to the full official version. Another', 'very useful addition is subtitles. Now,', 'every sound in your Mine

Output for cleaned raw transcript (list form)

sample 2: Extract text, remove captions

1. Transcript Text: (Just extract text and join)

[...'every sound in your Minecraft world can', 'show [music] as text on your screen. For', 'example, when you walk, it shows',...]

2. Transcript Text: (Also remove captions)

[...'every sound in your Minecraft world can', 'show  as text on your screen. For', 'example, when you walk, it shows',...]


In [ ]:
# Test chunking cleaned transcript

chunk_size = 10 # num of snippets from cleaned list
chunks = [] # list for chunks

for i in range(0, len(cleaned), chunk_size):
  chunk_snippets = cleaned[i:i+chunk_size] # get snippets for a chunk
  chunk = " ".join(chunk_snippets) # Join chunk snippets into a string
  chunks.append(chunk) # add chunk to list of chunks

# Test print chunks list
print(f"Transcript Text:\n{chunks}")

Transcript Text:
["So guys, Minecraft 26.0 is finally out, and this update brings more than 50 changes. Some are small fixes, some are visual upgrades, and some actually change how the game feels. Let's start with the menu. You probably remember when Mojang moved the settings button down in the main menu. A lot of players didn't like that change, and people even started making jokes about the browse", "add-ons option being more important than settings. Well, Mojang listened. The settings button is now back at the top where it feels natural. And not just that, when you open settings now, the entire layout has been redesigned. It looks cleaner, more modern, and much easier to navigate. The categories are better organized, and overall, it just feels more polished. Right now, it's in", "beta, but soon this redesign should come to the full official version. Another very useful addition is subtitles. Now, every sound in your Minecraft world can show  as text on your screen. For example, when

Output for chunking cleaned transcript

Transcript Text: (list of chunks, bolded text is ex of one chunk)

[**"So guys, Minecraft 26.0 is finally out, and this update brings more than 50 changes. Some are small fixes, some are visual upgrades, and some actually change how the game feels. Let's start with the menu. You probably remember when Mojang moved the settings button down in the main menu. A lot of players didn't like that change, and people even started making jokes about the browse",** "add-ons option being more important than settings. Well, Mojang listened. The settings button is now back at the top where it feels natural. And not just that, when you open settings now, the entire layout has been redesigned. It looks cleaner, more modern, and much easier to navigate. The categories are better organized, and overall, it just feels more polished. Right now, it's in", ...]

In [ ]:
"""
# Next: Test cleaned/chunked transcript in claim extraction step

# Function to extract claims
def claim_extraction(text):
    # Create message
    messages = [
        {"role": "system", "content": "You are a helpful assistant that extracts factual claims from text."},
        {"role": "user", "content": f"Extract all factual claims as bullet points.\n\nText: {text}"}
    ]
    # Send message to Qwen model
    output = pipe(messages, max_length=None)
    # Get claim from Qwen response
    result = output[0]["generated_text"][-1]["content"]
    return result

# Test extract claims for sample data
print(f"Claims Extracted:\n{claim_extraction(sample_data)}\n")
"""
